# BirdCLEF 2026 — Inference v23 (PerchGRU, Perch-only)
## ONNX Perch + Bidirectional GRU sequence model + Gaussian temporal smoothing

### Architecture:
- **Perch**: ONNX `perch_v2_cpu.onnx` via `onnxruntime` — no TF, no XLA issues
- **Classifier**: `PerchGRU` — BiGRU that processes the entire soundscape as a temporal sequence
  (all windows at once, so window N sees context from windows N-1 and N+1)
- **Post-processing**: Gaussian smoothing (σ=1.0) over the time axis

### Why this is better than v22:
- ONNX eliminates all TF-related runtime crashes → no more silent 0.5
- GRU captures temporal context across the soundscape
- Gaussian smoothing gives free +LB boost

### Required Kaggle inputs (internet OFF safe):
1. `birdclef-2026` (competition data)
2. `chiragggg/birdclef-2026-perch-onnx` OR `vyankteshdwivedi/perch-onnx-for-birdclef+2026` (Perch ONNX model)
3. `chiragggg/birdclef-2026-perch-weights-v23` (PerchGRU checkpoints)
4. **`chiragggg/onnxruntime-wheel`** — onnxruntime wheel for offline install (see setup below)

### One-time setup: create the onnxruntime wheel dataset
Run this **locally** (internet on) once, then upload:
```bash
# Download wheel for Kaggle's Linux x86_64 Python 3.11 environment
pip download onnxruntime --platform manylinux_2_17_x86_64 --python-version 311 --only-binary :all: --no-deps -d onnxruntime_wheel/
# Upload to Kaggle
kaggle datasets create -p onnxruntime_wheel/ --dir-mode zip
# dataset id: chiragggg/onnxruntime-wheel
```

In [ ]:
# === CELL 0: ENSURE ONNXRUNTIME IS AVAILABLE (no internet needed) ===
# Strategy: try direct import first; if missing, install from a pre-downloaded
# wheel stored as a Kaggle dataset (chiragggg/onnxruntime-wheel).
import sys, glob as _glob

try:
    import onnxruntime as _ort_test
    print(f'onnxruntime {_ort_test.__version__} already available ✅')
    del _ort_test
except ImportError:
    import subprocess
    _wheels = _glob.glob('/kaggle/input/onnxruntime-wheel/*.whl')
    if not _wheels:
        # Also search any dataset that might contain the wheel
        _wheels = _glob.glob('/kaggle/input/**/*.whl', recursive=True)
        _wheels = [w for w in _wheels if 'onnxruntime' in w]

    if _wheels:
        _whl = sorted(_wheels)[-1]   # pick the latest alphabetically
        print(f'Installing onnxruntime from wheel: {_whl}')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', _whl],
            check=True,
        )
        import onnxruntime as _ort_test
        print(f'onnxruntime {_ort_test.__version__} installed from wheel ✅')
        del _ort_test
    else:
        raise RuntimeError(
            'onnxruntime not found and no wheel dataset attached.\n'
            'Add dataset chiragggg/onnxruntime-wheel as input.\n'
            'See the Setup instructions in the header cell.'
        )


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, warnings, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from tqdm import tqdm
import gc

warnings.filterwarnings('ignore')

CFG = dict(
    folds          = 5,
    device         = 'cuda' if torch.cuda.is_available() else 'cpu',
    perch_sr       = 32000,
    perch_seconds  = 5,
    perch_emb_dim  = 1536,
    perch_batch    = 16,   # windows per ONNX forward pass
    gru_hidden     = 512,
    gru_layers     = 2,
    gauss_sigma    = 1.0,  # Gaussian smoothing over time axis
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160,000

device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device        : {device}')
print(f'onnxruntime   : {ort.__version__}')
print(f'Gauss sigma   : {CFG["gauss_sigma"]}')
print(f'Providers     : {ort.get_available_providers()}')

In [ ]:
# === CELL 2: PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV = _first_existing(
    '/kaggle/input/birdclef-2026/taxonomy.csv',
    '/kaggle/input/competitions/birdclef-2026/taxonomy.csv',
)
TEST_AUDIO = _first_existing(
    '/kaggle/input/birdclef-2026/test_soundscapes',
    '/kaggle/input/competitions/birdclef-2026/test_soundscapes',
)
SAMPLE_SUB = _first_existing(
    '/kaggle/input/birdclef-2026/sample_submission.csv',
    '/kaggle/input/competitions/birdclef-2026/sample_submission.csv',
)
PERCH_CKPT_DIR = _first_existing(
    '/kaggle/input/birdclef-2026-perch-weights-v23',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v23',
    '/kaggle/working',
)

# ONNX model: try our own upload first, then the public one from vyankteshdwivedi
print('Searching for Perch ONNX model ...')
_onnx_candidates = [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    # Public dataset from top-0.944 notebook (vyankteshdwivedi/perch-onnx-for-birdclef+2026)
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/model.onnx',
]
ONNX_PATH = None
for _c in _onnx_candidates:
    if os.path.exists(_c):
        ONNX_PATH = _c
        print(f'  Found: {ONNX_PATH}')
        break
if ONNX_PATH is None:
    # List all .onnx files available
    import glob
    _found = glob.glob('/kaggle/input/**/*.onnx', recursive=True)
    print(f'  No .onnx found at expected paths. All .onnx files in /kaggle/input: {_found}')
    print('  Add chiragggg/birdclef-2026-perch-onnx OR vyankteshdwivedi/perch-onnx-for-birdclef+2026 as input.')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f'TAXONOMY     : {TAXONOMY_CSV}')
print(f'TEST_AUDIO   : {TEST_AUDIO}  (empty at commit time)')
print(f'CKPT_DIR     : {PERCH_CKPT_DIR}')
print(f'ONNX_PATH    : {ONNX_PATH}')
print(f'Species      : {n_classes}')

In [ ]:
# === CELL 3: PERCHGRU DEFINITION + LOAD CHECKPOINTS ===
class PerchGRU(nn.Module):
    def __init__(self, n_classes: int, emb_dim: int = 1536,
                 hidden: int = 512, n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=512, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )
    def forward(self, x):
        # x: (B, T, 1536)  or  (1, T, 1536) at inference
        z = self.proj(x)
        h, _ = self.gru(z)
        return self.head(h)  # (B, T, n_classes)


gru_models = []
missing    = []
for fold_idx in range(CFG['folds']):
    ckpt = Path(PERCH_CKPT_DIR) / f'perch_gru_v23_fold{fold_idx}.pt'
    if not ckpt.exists():
        missing.append(str(ckpt))
        continue
    m = PerchGRU(n_classes, CFG['perch_emb_dim'], CFG['gru_hidden'], CFG['gru_layers']).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    m.eval()
    gru_models.append(m)
    print(f'   ✅ {ckpt.name}')

if missing:
    print(f'\n⚠️  {len(missing)} checkpoint(s) not found:')
    for p in missing:
        print(f'   {p}')

print(f'\n✅ {len(gru_models)} PerchGRU models loaded')
if not gru_models:
    raise RuntimeError('No PerchGRU checkpoints. Add birdclef-2026-perch-weights-v23 as input.')

In [ ]:
# === CELL 4: LOAD ONNX SESSION + SANITY CHECK ===
_ort_session  = None
_onnx_inp     = None
_onnx_emb_key = None
_onnx_ready   = False

if ONNX_PATH is None:
    print('ERROR: ONNX file not found (see Cell 2). Predictions will be 0.5.')
else:
    try:
        _sess_opts = ort.SessionOptions()
        _sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        _sess_opts.intra_op_num_threads = os.cpu_count() or 4
        _ort_session = ort.InferenceSession(
            ONNX_PATH,
            sess_options=_sess_opts,
            providers=['CPUExecutionProvider'],
        )
        _onnx_inp = _ort_session.get_inputs()[0].name

        # Auto-detect 1536-d output
        for out in _ort_session.get_outputs():
            if out.shape and out.shape[-1] == 1536:
                _onnx_emb_key = out.name
                break
        if _onnx_emb_key is None:
            # Fall back: run a test pass to find 1536-d output
            _test = np.zeros((1, CFG['perch_target']), dtype=np.float32)
            _outs = _ort_session.run(None, {_onnx_inp: _test})
            _names = [o.name for o in _ort_session.get_outputs()]
            for _nm, _ov in zip(_names, _outs):
                if _ov.ndim >= 2 and _ov.shape[-1] == 1536:
                    _onnx_emb_key = _nm
                    break
        assert _onnx_emb_key, 'Cannot find 1536-d output in ONNX model.'

        # Check model_info.json if present
        _info_json = Path(ONNX_PATH).parent / 'model_info.json'
        if _info_json.exists():
            import json
            _info = json.load(open(_info_json))
            _onnx_inp     = _info.get('input_name', _onnx_inp)
            _onnx_emb_key = _info.get('embedding_output_name', _onnx_emb_key)
            print(f'Loaded model_info.json: input={_onnx_inp!r}, emb={_onnx_emb_key!r}')

        # Sanity check with sine wave
        _test = np.sin(
            2 * np.pi * 440 * np.linspace(0, 5, CFG['perch_target'])
        ).astype(np.float32)[None]
        _all_outs  = _ort_session.run(None, {_onnx_inp: _test})
        _out_names = [o.name for o in _ort_session.get_outputs()]
        _emb        = _all_outs[_out_names.index(_onnx_emb_key)]
        if _emb.ndim == 3:
            _emb = _emb.mean(axis=1)
        _emb = _emb[0]
        assert _emb.shape == (1536,), f'Bad shape: {_emb.shape}'
        assert _emb.std() >= 0.05, f'Near-zero std={_emb.std():.4f}. ONNX model may be corrupt.'

        print(f'✅ ONNX session ready.')
        print(f'   input ={_onnx_inp!r}  emb_output={_onnx_emb_key!r}')
        print(f'   Sanity: shape={_emb.shape}  mean={_emb.mean():.4f}  std={_emb.std():.4f}')
        del _test, _all_outs, _emb
        _onnx_ready = True

    except Exception as _e:
        traceback.print_exc()
        print(f'\nERROR loading ONNX: {_e}')
        _ort_session = None

if not _onnx_ready:
    print('\n⚠️  ONNX NOT READY — predictions will be 0.5')


In [ ]:
# === CELL 5: PREDICTION FUNCTION ===
_use_amp     = (device.type == 'cuda')
_out_names   = [o.name for o in _ort_session.get_outputs()] if _ort_session else []
_emb_idx     = _out_names.index(_onnx_emb_key) if _onnx_emb_key in _out_names else 0


def _compute_embs_onnx(audio_path: str, end_secs_list: list) -> dict:
    """
    Compute Perch embeddings for each requested 5s window via ONNX.
    Returns {end_s: np.ndarray(1536,)}. Missing windows are omitted.
    """
    result = {}
    if _ort_session is None or not end_secs_list:
        return result

    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as _e:
        print(f'   [WARN] Audio load failed: {audio_path}: {_e}')
        return result

    # Extract windows
    clips = []
    for es in end_secs_list:
        end_samp   = int(es * CFG['perch_sr'])
        start_samp = max(0, end_samp - CFG['perch_target'])
        clip       = y[start_samp:end_samp]
        if len(clip) < CFG['perch_target']:
            clip = np.pad(clip, (0, CFG['perch_target'] - len(clip)))
        clips.append(clip)

    # Batch ONNX forward passes
    all_embs = []
    for i in range(0, len(clips), CFG['perch_batch']):
        batch = np.stack(clips[i : i + CFG['perch_batch']])
        try:
            outs = _ort_session.run(None, {_onnx_inp: batch})
            embs = outs[_emb_idx]       # (B, 1536) or (B, T_frames, 1536)
            if embs.ndim == 3:
                embs = embs.mean(axis=1)
            all_embs.append(embs.astype(np.float32))
        except Exception as _e:
            print(f'   [ERROR] ONNX forward: {_e}')
            return result  # return what we have

    if not all_embs:
        return result

    all_embs_np = np.vstack(all_embs)
    _batch_std  = all_embs_np.std()
    if _batch_std < 0.05:
        print(f'   [WARN] Near-zero embeddings for {Path(audio_path).stem} (std={_batch_std:.4f})')

    for es, emb in zip(end_secs_list, all_embs_np):
        result[es] = emb
    return result


def predict_soundscape_v23(audio_path: str, end_seconds: list) -> np.ndarray:
    """
    Full pipeline for one soundscape:
      1. Compute all window embeddings via ONNX            -> (T, 1536)
      2. Run through PerchGRU ensemble (all windows at once) -> (T, n_classes)
      3. Apply Gaussian smoothing over T                   -> (T, n_classes)
    Returns (T, n_classes) probabilities.
    """
    T       = len(end_seconds)
    neutral = np.full((T, n_classes), 0.5, dtype=np.float32)

    if not gru_models or not _onnx_ready:
        return neutral

    # Step 1: embeddings
    embs_map   = _compute_embs_onnx(audio_path, end_seconds)
    emb_list   = [
        embs_map.get(es, np.zeros(CFG['perch_emb_dim'], dtype=np.float32))
        for es in end_seconds
    ]
    # (1, T, 1536) — single batch, GRU sees full soundscape sequence
    emb_seq = torch.from_numpy(np.stack(emb_list)).float().unsqueeze(0).to(device)  # (1, T, 1536)

    # Step 2: GRU ensemble
    all_probs = []
    for m in gru_models:
        with torch.inference_mode(), autocast(enabled=_use_amp):
            logits = m(emb_seq).float()         # (1, T, n_classes)
            probs  = torch.sigmoid(logits)[0].cpu().numpy()  # (T, n_classes)
        all_probs.append(probs)
    probs_mean = np.mean(all_probs, axis=0).astype(np.float32)  # (T, n_classes)

    # Step 3: Gaussian smoothing over T
    if T > 1 and CFG['gauss_sigma'] > 0:
        probs_mean = gaussian_filter1d(
            probs_mean.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0
        ).astype(np.float32)

    return probs_mean


print('✅ predict_soundscape_v23() defined')
print(f'   ONNX ready  : {_onnx_ready}')
print(f'   GRU models  : {len(gru_models)}')
print(f'   Gauss sigma : {CFG["gauss_sigma"]}')

In [ ]:
# === CELL 6: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
sample_sub = sample_sub.copy()
sample_sub['_sc_id'] = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Submission rows: {len(sample_sub)}')

all_row_ids    = []
all_probs_list = []
missing_audio  = 0
error_count    = 0

for sc_id, grp in tqdm(sample_sub.groupby('_sc_id'), desc='Soundscapes', unit='file'):
    row_ids = [str(r) for r in grp['row_id']]

    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc_id}{ext}'
        if c.exists():
            audio_path = str(c)
            break

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        end_seconds = [int(rid.rsplit('_', 1)[-1]) for rid in row_ids]
    except Exception as _e:
        print(f'WARNING: parse end_seconds failed for {sc_id}: {_e}')
        error_count += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        probs = predict_soundscape_v23(audio_path, end_seconds)
    except Exception as _e:
        print(f'WARNING: prediction failed for {sc_id}: {_e}')
        traceback.print_exc()
        error_count += 1
        probs = np.full((len(row_ids), n_classes), 0.5, dtype=np.float32)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs)

if missing_audio:
    print(f'\n⚠️  {missing_audio} soundscape(s) had no audio (expected if test set empty at commit time)')
if error_count:
    print(f'⚠️  {error_count} prediction failure(s) — used 0.5')
print(f'\n✅ Generated {len(all_row_ids)} rows')

In [ ]:
# === CELL 7: BUILD & SAVE SUBMISSION ===
probs_matrix = np.concatenate(all_probs_list, axis=0)

_mean_p = probs_matrix.mean()
_std_p  = probs_matrix.std()
if abs(_mean_p - 0.5) < 0.001 and _std_p < 0.01:
    print(f'\n⚠️  WARNING: all predictions near 0.5 (mean={_mean_p:.4f}, std={_std_p:.4f})')
    print('   Check Cell 4 ONNX output and Cell 3 checkpoint loading.')
else:
    print(f'✅ Prediction distribution healthy: mean={_mean_p:.4f}, std={_std_p:.4f}')

sub_df = pd.DataFrame(probs_matrix, columns=species)
sub_df.insert(0, 'row_id', all_row_ids)
sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[sample_cols]

out_path = '/kaggle/working/submission.csv'
sub_df.to_csv(out_path, index=False)
print(f'✅ Submission saved: {out_path}  shape={sub_df.shape}')
print(sub_df.head(3))